# Setup

## Import modules

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
import torch

# Import pipeline modules
from scoring import score_image
from selection import select
from metrics import MetricsTracker, image_metrics_dict
from generation import generate_unconditional
from utils import save_image, save_image_batch, set_seed

# Config
OUTPUT_DIR = Path("outputs")
IMAGES_DIR = OUTPUT_DIR / "images"
METRICS_CSV = OUTPUT_DIR / "metrics.csv"

# Experiment parameters
NUM_PROMPTS = 5
ROUNDS = 2        # r
BATCH_SIZE = 4    # B
SEED = 42

# Chosen strategies
SELECTION_STRATEGY = "argmax"

# Baseline scoring rubric (example)
# CLIP_RUBRIC = {"type": "clip", "text": "A photorealistic portrait of a dog", "weight": 1.0}
BRIGHTNESS_RUBRIC = {"type": "brightness", "weight": 1.0}

set_seed(SEED)


## Load diffusion model

In [ ]:
# Cell 2: load diffusion pipeline (adjust model_name as needed)
from diffusers import StableDiffusionPipeline

MODEL_NAME = "runwayml/stable-diffusion-v2-2"  # replace with SD3 HF model if available
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16 if device=="cuda" else torch.float32)
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # memory-friendly

## Load dataset

In [ ]:
# Mock dataset
dataset = [
    {"prompt_id": f"p{i}+1", 
    "prompt": f"A photograph of {i+1} dogs",
    "rubric": BRIGHTNESS_RUBRIC}
    for i in range(NUM_PROMPTS)
]

# Perform simulation

In [ ]:
metrics = MetricsTracker()

for item in tqdm(dataset, desc="prompts"):
    prompt_id = item["prompt_id"]
    prompt = item["prompt"]
    rubric = item["rubric"]

    print(f"New item: prompt_id={prompt_id}, prompt={prompt}, rubric={rubric}")

    prompt_outdir = IMAGES_DIR / prompt_id
    prompt_outdir.mkdir(parents=True, exist_ok=True)

    # Optional: save the prompt text
    (prompt_outdir / "prompt.txt").write_text(prompt)

    for round in range(ROUNDS):
        print(f"\tRound {round}")

        # Generate images using our method
        batch_images: list[Image.Image] = generate_unconditional(
            pipe,
            prompt,
            batch_size=BATCH_SIZE,
            num_inference_steps=20,
            guidance_scale=7.5,
            width=512,
            height=512,
        )
        
        # Score and record metrics for each image in the batch
        scores = []
        for image_idx, image in enumerate(batch_images):
            score = score_image(image, rubric)
            scores.append(score)
            
            row = image_metrics_dict(
                prompt_id,
                round,
                image_idx   ,
                image,
                score=score,
                chosen=False, # will be overridden below
            )
            metrics.append(row)

        # Select favorite (index)
        chosen_idx = select(scores, strategy=SELECTION_STRATEGY)
        
        print(f"\t\tScores: {scores}")
        print(f"\t\tChosen index (using {SELECTION_STRATEGY}): {chosen_idx}")
        
        # Mark chosen in metrics store (the last appended rows correspond to this batch)
        # (Simplest way: update the last B rows)
        df = metrics.to_dataframe()
        last_idx = len(df) - 1
        
        # Update chosen flag in the internal list
        # Because MetricsStore stores raw rows, we update those rows directly:
        for rel_i in range(BATCH_SIZE):
            metrics._rows[-(BATCH_SIZE - rel_i)]["chosen"] = (rel_i == chosen_idx)

        # Save batch images for this round, with a sub-directory for each round
        round_outdir = prompt_outdir / f"round{round}"
        round_outdir.mkdir(parents=True, exist_ok=True)

        names = save_image_batch(batch_images, str(round_outdir), name_prefix=f"image{image_idx}")

        # Save chosen as separate file
        chosen_path = prompt_outdir / f"round{round}_chosen.png"
        batch_images[chosen_idx].save(chosen_path)

# Save metrics
metrics.save_csv(str(METRICS_CSV))
print("Saved metrics to", METRICS_CSV)
